# 05 — Inspect data distributions (robotics fitness)

Use this while learning full-scale robotics to answer:
1. What topics / rates are in my logs?
2. How are joints, VLA tokens, and commands distributed?
3. Does this data **apply** to real robot control learning (limits, smoothness, coverage)?

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "_lib"))
from bootstrap import setup
ROOT = setup()

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from s2r.core.config import load_config
from s2r.experiments.inspect_data import (
    inspect_dataset, inspect_benchmark_coverage, save_report, load_jsonl_rows, extract_series
)
from s2r.experiments.paths import RAW, ensure_experiment_dirs

ensure_experiment_dirs()
cfg = load_config(ROOT / "config" / "default.yaml")
report = inspect_dataset(RAW, joint_limits=cfg["robot"]["joint_limits"])
report["benchmark_coverage"] = inspect_benchmark_coverage()
path = save_report(report)
print("wrote", path)
report["rates_hz"], report["robotics_fitness"]

In [ ]:
fit = report["robotics_fitness"]
print("APPLIES TO ROBOTICS?", fit["applies"], "score=", round(fit["score"], 3))
print("checklist")
display(pd.Series(fit["checklist"]))
print("reasons")
for r in fit["reasons"]:
    print(" -", r)
print("warnings")
for w in fit["warnings"]:
    print(" -", w)

In [ ]:
pd.DataFrame([report["topic_counts"]]).T.rename(columns={0: "count"})

In [ ]:
# Joint / action / command distributions
def dist_df(key):
    return pd.DataFrame(report["distributions"][key])

display(dist_df("joint_pos"))
display(dist_df("action_token"))
display(dist_df("joint_cmd").head(10))
pd.DataFrame([report["distributions"]["latency_ms"], report["distributions"]["cmd_jerk_proxy"]])

In [ ]:
rows = load_jsonl_rows(RAW)
series = extract_series(rows)
cmds = series["cmds"]
acts = series["actions"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
if cmds.size:
    axes[0].hist(cmds[:, 0], bins=40, color="#3dd6c6")
    axes[0].set_title("joint_cmd j0")
if acts.size:
    axes[1].hist(acts[:, 0], bins=40, color="#5d8cff")
    axes[1].set_title("action_token j0")
if cmds.shape[0] > 5:
    axes[2].plot(cmds[:500, 0], label="cmd0")
    if acts.shape[0] > 1:
        # overlay sparse tokens roughly
        axes[2].plot(np.linspace(0, min(500, len(cmds)-1), num=min(len(acts), 50)), acts[:50, 0], "o", ms=3, label="tokens")
    axes[2].legend(); axes[2].set_title("trajectory preview")
plt.tight_layout(); plt.show()

In [ ]:
pd.DataFrame(report["benchmark_coverage"]["tasks"])

## How to read this (learner notes)

| Signal | Healthy robotics range |
|---|---|
| `state` / `joint_cmd` rate | tens–hundreds Hz for arms |
| `action_token` (VLA) | ~1–5 Hz sparse semantic actions |
| joints within limits | mostly true |
| command jerk p95 | not huge (smooth enough for hardware) |
| decisions/mission | present if you care about long-horizon tasks |

If `robotics_applies=False`, collect more episodes with the live pipeline before training ESN or benchmarking VLA.